In [1]:
# Install & Import Libraries
import pandas as pd
import numpy as np
import re
import random
import json
import torch
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
 
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import ndcg_score
 
warnings.filterwarnings('ignore')
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

ImportError: DLL load failed while importing lib: An Application Control policy has blocked this file.

In [ ]:
# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 5060 Ti


In [ ]:
#  Load Data

RESUME_PATH = 'live_resume/Resume/Resume.csv'
JOB_PATH = 'linkedin_job/filtered_job_postings.csv'
 
resume_df = pd.read_csv(RESUME_PATH)
job_df = pd.read_csv(JOB_PATH)
 
job_text_col = 'job_description'
job_cat_col = 'Category'
 
print(f"Resumes loaded: {len(resume_df)} rows")
print(f"Jobs loaded: {len(job_df)} rows")
print(f"\nResume categories ({resume_df['Category'].nunique()}): {sorted(resume_df['Category'].unique())}")
print(f"Job categories ({job_df[job_cat_col].nunique()}): {sorted(job_df[job_cat_col].unique())}")

Resumes loaded: 2484 rows
Jobs loaded: 101492 rows

Resume categories (24): ['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER']
Job categories (24): ['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER']


In [ ]:
#  Text Preprocessing (lighter than LSTM — transformer handles raw text better)

 
stop_words = set(stopwords.words('english'))
 
def clean_text(text):
    """Light cleaning for Sentence Transformer — preserve more structure than LSTM version."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<[^>]+>', ' ', text)         # Remove HTML tags
    text = re.sub(r'http\S+|www\S+', ' ', text)   # Remove URLs
    text = re.sub(r'\S+@\S+', ' ', text)           # Remove emails
    text = re.sub(r'[^a-zA-Z0-9\s\.\,\-]', ' ', text)  # Keep letters, numbers, basic punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text
 
def truncate_text(text, max_words=256):
    """Truncate to max_words — MiniLM has a 256 token limit for best performance."""
    words = text.split()[:max_words]
    return ' '.join(words)
 
print("Cleaning resume texts...")
resume_df['clean_text'] = resume_df['Resume_str'].apply(clean_text).apply(truncate_text)
 
print("Cleaning job description texts...")
job_df['clean_text'] = job_df[job_text_col].apply(clean_text).apply(truncate_text)
 
# Drop empty rows
resume_df = resume_df[resume_df['clean_text'].str.len() > 0].reset_index(drop=True)
job_df = job_df[job_df['clean_text'].str.len() > 0].reset_index(drop=True)
 
print(f"\nAfter cleaning — Resumes: {len(resume_df)}, Jobs: {len(job_df)}")
print(f"Sample resume (first 150 chars): {resume_df['clean_text'].iloc[0][:150]}")
print(f"Sample job (first 150 chars): {job_df['clean_text'].iloc[0][:150]}")

Cleaning resume texts...
Cleaning job description texts...

After cleaning — Resumes: 2483, Jobs: 101486
Sample resume (first 150 chars): HR ADMINISTRATOR MARKETING ASSOCIATE HR ADMINISTRATOR Summary Dedicated Customer Service Manager with 15 years of experience in Hospitality and Custom
Sample job (first 150 chars): The National Exemplar is accepting applications for an Assistant Restaurant Manager. We offer highly competitive wages, healthcare, paid time off, com


In [ ]:
# Define Related Categories 

 
RELATED_CATEGORIES = {
    'Information-Technology': ['Engineering', 'Digital-Media', 'BPO', 'Consultant'],
    'Engineering': ['Information-Technology', 'Automobile', 'Aviation', 'Construction'],
    'Healthcare': ['Fitness', 'Agriculture'],
    'Finance': ['Accountant', 'Banking', 'Business-Development', 'Consultant'],
    'Accountant': ['Finance', 'Banking', 'Business-Development'],
    'Banking': ['Finance', 'Accountant', 'Business-Development'],
    'Sales': ['Business-Development', 'Public-Relations', 'Advocate', 'Consultant'],
    'Business-Development': ['Sales', 'Finance', 'Consultant', 'Public-Relations'],
    'Arts': ['Designer', 'Apparel', 'Digital-Media', 'Public-Relations'],
    'Designer': ['Arts', 'Apparel', 'Digital-Media'],
    'Digital-Media': ['Arts', 'Designer', 'Information-Technology', 'Public-Relations'],
    'Apparel': ['Arts', 'Designer'],
    'Aviation': ['Engineering', 'Automobile'],
    'Automobile': ['Engineering', 'Aviation', 'Construction'],
    'Construction': ['Engineering', 'Automobile'],
    'Teacher': ['Consultant', 'Public-Relations', 'HR'],
    'HR': ['Business-Development', 'Consultant', 'Public-Relations', 'Teacher'],
    'Advocate': ['Sales', 'Public-Relations', 'Consultant'],
    'Consultant': ['Business-Development', 'Information-Technology', 'Finance', 'HR'],
    'Chef': ['Agriculture'],
    'Agriculture': ['Chef', 'Healthcare', 'Fitness'],
    'Fitness': ['Healthcare', 'Agriculture'],
    'BPO': ['Information-Technology', 'Sales', 'HR'],
    'Public-Relations': ['Sales', 'HR', 'Arts', 'Digital-Media', 'Business-Development'],
}
 
def get_similarity_label(cat1, cat2):
    """Return similarity score based on category relationship."""
    if cat1 == cat2:
        return 0.9
    elif cat2 in RELATED_CATEGORIES.get(cat1, []):
        return 0.5
    else:
        return 0.1

In [ ]:
# Generate Training Pairs (10,000-15,000 for fine-tuning)

 
print("Generating training pairs for fine-tuning...")
 
resume_by_cat = resume_df.groupby('Category')['clean_text'].apply(list).to_dict()
job_by_cat = job_df.groupby(job_cat_col)['clean_text'].apply(list).to_dict()
 
categories = sorted(set(resume_by_cat.keys()) & set(job_by_cat.keys()))
print(f"Overlapping categories: {len(categories)}")
 
pairs = []  # (resume_text, job_text, similarity_score)
 
# --- Same category pairs (~5,000) ---
TARGET_SAME = 5000
same_per_cat = TARGET_SAME // len(categories)
for cat in categories:
    resumes = resume_by_cat[cat]
    jobs = job_by_cat[cat]
    for _ in range(same_per_cat):
        r = random.choice(resumes)
        j = random.choice(jobs)
        pairs.append((r, j, 0.9))
 
# --- Related category pairs (~4,000) ---
TARGET_RELATED = 4000
related_pairs_pool = []
for cat in categories:
    related_cats = [c for c in RELATED_CATEGORIES.get(cat, []) if c in job_by_cat]
    for rel_cat in related_cats:
        related_pairs_pool.append((cat, rel_cat))
 
if related_pairs_pool:
    rel_per_pair = max(1, TARGET_RELATED // len(related_pairs_pool))
    for res_cat, job_cat_name in related_pairs_pool:
        resumes = resume_by_cat[res_cat]
        jobs = job_by_cat[job_cat_name]
        for _ in range(rel_per_pair):
            r = random.choice(resumes)
            j = random.choice(jobs)
            pairs.append((r, j, 0.5))
 
# --- Unrelated category pairs (~5,000) ---
TARGET_UNRELATED = 5000
unrelated_count = 0
while unrelated_count < TARGET_UNRELATED:
    cat1 = random.choice(categories)
    cat2 = random.choice(categories)
    if cat1 != cat2 and cat2 not in RELATED_CATEGORIES.get(cat1, []):
        r = random.choice(resume_by_cat[cat1])
        j = random.choice(job_by_cat[cat2])
        pairs.append((r, j, 0.1))
        unrelated_count += 1
 
random.shuffle(pairs)
print(f"\nTotal training pairs generated: {len(pairs)}")
labels = [p[2] for p in pairs]
print(f"  Same category (0.9):    {labels.count(0.9)}")
print(f"  Related category (0.5): {labels.count(0.5)}")
print(f"  Unrelated (0.1):        {labels.count(0.1)}")

Generating training pairs for fine-tuning...
Overlapping categories: 24

Total training pairs generated: 13992
  Same category (0.9):    4992
  Related category (0.5): 4000
  Unrelated (0.1):        5000


In [ ]:
# Prepare Data for Sentence Transformers
 
# Split: 80% train, 10% val, 10% test
train_pairs, temp_pairs = train_test_split(pairs, test_size=0.2, random_state=SEED)
val_pairs, test_pairs = train_test_split(temp_pairs, test_size=0.5, random_state=SEED)
 
print(f"Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}")
 
# Convert to InputExample format (required by sentence-transformers)
train_examples = [
    InputExample(texts=[r, j], label=float(score))
    for r, j, score in train_pairs
]
 
# Validation evaluator — uses cosine similarity to evaluate during training
val_sentences1 = [p[0] for p in val_pairs]
val_sentences2 = [p[1] for p in val_pairs]
val_scores = [p[2] for p in val_pairs]
 
evaluator = evaluation.EmbeddingSimilarityEvaluator(
    val_sentences1, val_sentences2, val_scores,
    name='resume-job-validation'
)
 
# Training DataLoader
BATCH_SIZE = 32
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
 
print(f"\nTraining batches per epoch: {len(train_dataloader)}")
print("Data preparation complete.")
 

Train: 11193 | Val: 1399 | Test: 1400

Training batches per epoch: 350
Data preparation complete.


In [ ]:
#  Load Pre-trained Model
 
MODEL_NAME = 'all-MiniLM-L6-v2'
 
print(f"Loading pre-trained model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
 
print(f"\nModel loaded successfully!")
print(f"  Max sequence length: {model.max_seq_length}")
print(f"  Embedding dimension: {model.get_sentence_embedding_dimension()}")
 
# Test it with a quick encoding
test_embed = model.encode("software engineer with python experience")
print(f"  Test embedding shape: {test_embed.shape}")
print(f"  Test embedding sample: {test_embed[:5]}")

Loading pre-trained model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Model loaded successfully!
  Max sequence length: 256
  Embedding dimension: 384
  Test embedding shape: (384,)
  Test embedding sample: [-0.0684168   0.00592654  0.00516982  0.07372809 -0.03503487]


In [ ]:
# Evaluate BEFORE Fine-Tuning (Pre-trained baseline)

 
print("Evaluating pre-trained model BEFORE fine-tuning...")
print("(This gives us a baseline to measure improvement)\n")
 
# Encode all test pairs
test_sentences1 = [p[0] for p in test_pairs]
test_sentences2 = [p[1] for p in test_pairs]
test_labels = np.array([p[2] for p in test_pairs])
 
# Get embeddings
embeddings1 = model.encode(test_sentences1, show_progress_bar=True, batch_size=64)
embeddings2 = model.encode(test_sentences2, show_progress_bar=True, batch_size=64)
 
# Compute cosine similarity
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
 
pre_ft_predictions = np.array([
    cos_sim([e1], [e2])[0][0]
    for e1, e2 in zip(embeddings1, embeddings2)
])
 
# Bucket accuracy (same as Part 1)
buckets = np.array([0.1, 0.5, 0.9])
pred_buckets = buckets[np.argmin(np.abs(pre_ft_predictions[:, None] - buckets[None, :]), axis=1)]
true_buckets = buckets[np.argmin(np.abs(test_labels[:, None] - buckets[None, :]), axis=1)]
pre_ft_accuracy = np.mean(pred_buckets == true_buckets)
 
print(f"Pre-trained model (BEFORE fine-tuning):")
print(f"  Bucket Accuracy: {pre_ft_accuracy:.4f} ({pre_ft_accuracy*100:.1f}%)")
print(f"  Avg predicted similarity: {pre_ft_predictions.mean():.4f}")
print(f"  Predictions range: [{pre_ft_predictions.min():.4f}, {pre_ft_predictions.max():.4f}]")

Evaluating pre-trained model BEFORE fine-tuning...
(This gives us a baseline to measure improvement)



Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Pre-trained model (BEFORE fine-tuning):
  Bucket Accuracy: 0.3521 (35.2%)
  Avg predicted similarity: 0.3219
  Predictions range: [-0.0044, 0.7137]


In [ ]:
# CELL 9 — Fine-Tune the Model

 
EPOCHS = 5
WARMUP_STEPS = int(len(train_dataloader) * EPOCHS * 0.1)  
OUTPUT_DIR = 'fine_tuned_minilm'
 
# Loss function: Cosine Similarity Loss
# Trains the model so that cosine_sim(embed(resume), embed(job)) ≈ target_score
train_loss = losses.CosineSimilarityLoss(model)
 
print(f"Fine-tuning configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Training examples: {len(train_examples)}")
print(f"  Warmup steps: {WARMUP_STEPS}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"\nStarting fine-tuning...\n")
 
start_time = datetime.now()
 
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    output_path=OUTPUT_DIR,
    evaluation_steps=len(train_dataloader) // 2,  # Evaluate twice per epoch
    save_best_model=True,
    show_progress_bar=True,
)
 
elapsed = datetime.now() - start_time
print(f"\nFine-tuning complete! Time: {elapsed}")
print(f"Best model saved to: {OUTPUT_DIR}/")

Fine-tuning configuration:
  Epochs: 5
  Batch size: 32
  Training examples: 11193
  Warmup steps: 175
  Output directory: fine_tuned_minilm

Starting fine-tuning...



NameError: name 'Dataset' is not defined